# Argus — plate detector training (Kaggle GPU)

Trains the single-class Indian license-plate detector. This is the **primary**
training path; the local CPU fallback (`--cpu`) exists only if Kaggle is
unavailable and is 15-25x slower.

## Before you run anything

| Setting (right panel) | Value |
|---|---|
| **Accelerator** | GPU T4 x2 (or P100) |
| **Internet** | **On** — required to `pip install` and clone the repo |
| **Input → Add Data** | an Indian number plate dataset in **YOLO format** |

A new or unverified Kaggle account cannot enable GPU or Internet — both toggles
stay greyed out. Fix it once at kaggle.com/settings → Phone Verification.

Runtime: ~20-40 s/epoch, so 50 epochs is roughly 20-35 minutes.

## Step 1 — Configure

Click the attached dataset in the right panel to see its mount path, and paste
it into `SRC`.

In [ ]:
SRC     = "/kaggle/input/CHANGE-ME"   # <- attached dataset root
EPOCHS  = 50
SUBSET  = 3000    # train images. On GPU you can afford 15000 — raise it if mAP disappoints.
VAL     = 400

## Step 2 — Confirm the GPU and install

`lap` is BoT-SORT's assignment solver. Installing it here rather than letting
Ultralytics AutoUpdate it mid-run avoids a pip install firing in the middle of
training.

In [ ]:
!nvidia-smi
!pip -q install ultralytics lap

## Step 3 — Clone the repo

The notebook calls `ml/prepare_dataset.py` and `ml/train_plate.py` from the
repo. **No training code is duplicated here**, so the Kaggle path and the local
path cannot drift apart.

In [ ]:
%cd /kaggle/working
!rm -rf Argus && git clone -q https://github.com/Deeptanshu789/Argus.git
%cd /kaggle/working/Argus

## Step 4 — Inspect the dataset BEFORE training

Do not skip this. `prepare_dataset.py` reads YOLO format: one `.txt` per image,
each line `class cx cy w h` with normalized coordinates.

Many Kaggle plate datasets ship **Pascal VOC `.xml`** annotations instead. Those
produce zero image/label pairs and a confusing failure. Find out in ten seconds
here rather than inside a training run.

In [ ]:
import subprocess, pathlib

print("--- top-level layout ---")
!ls {SRC}

txt = subprocess.run(f'find {SRC} -name "*.txt" -not -name "classes.txt"',
                     shell=True, capture_output=True, text=True).stdout.split()
xml = subprocess.run(f'find {SRC} -name "*.xml"', shell=True,
                     capture_output=True, text=True).stdout.split()
imgs = subprocess.run(f'find {SRC} \\( -name "*.jpg" -o -name "*.png" -o -name "*.jpeg" \\)',
                      shell=True, capture_output=True, text=True).stdout.split()

print(f"\nimages: {len(imgs)}   YOLO .txt labels: {len(txt)}   Pascal VOC .xml: {len(xml)}")

if txt:
    print(f"\nsample label ({txt[0]}):")
    print(pathlib.Path(txt[0]).read_text()[:200])
    print("\nEach line must be: class cx cy w h, all of cx/cy/w/h between 0 and 1,")
    print("and class must be 0 (single class). If class is not 0, prepare_dataset")
    print("still copies it and training will silently learn the wrong thing.")

assert txt, (
    "No YOLO .txt labels found."
    + (" This dataset is Pascal VOC (.xml) — pick a different one, or export from "
       "Roboflow Universe in YOLOv8 format." if xml else "")
)
print("\nOK — YOLO format.")

## Step 5 — Prepare

Finds every image/label pair at any depth, drops unlabelled images, splits, and
writes a valid `data.yaml`. Handles both flat and pre-split layouts.

In [ ]:
!python ml/prepare_dataset.py --src "{SRC}" --subset {SUBSET} --val {VAL}
!cat datasets/plates/data.yaml

## Step 6 — Train

GPU defaults from `ml/train_plate.py`: `imgsz=640, batch=32, amp=True,
freeze=0`.

Note `freeze=0` — the backbone is **not** frozen. Freezing it is a CPU
concession that costs accuracy, and on a T4 there is no reason to pay it.

In [ ]:
!python ml/train_plate.py --epochs {EPOCHS} --device 0

## Step 7 — Read the score

**Go/no-go bar: `mAP50 >= 0.85`.**

In [ ]:
import csv, pathlib

R = pathlib.Path("runs/detect/plate")
rows = list(csv.DictReader(open(R / "results.csv")))
last = {k.strip(): v for k, v in rows[-1].items()}
m50   = float(last["metrics/mAP50(B)"])
m5095 = float(last["metrics/mAP50-95(B)"])
print(f"epochs run : {len(rows)}")
print(f"mAP50      : {m50:.3f}")
print(f"mAP50-95   : {m5095:.3f}\n")

if m50 >= 0.85:
    print("PASS — export the weights (Step 8).")
elif m50 >= 0.70:
    print("MARGINAL — raise SUBSET toward 15000 and re-run. A GPU run costs minutes.")
else:
    print("FAIL — this is almost certainly a DATA problem, not a training one.")
    print("Single-class, tight-boxed plates reach 0.85 readily; more epochs will")
    print("not fix a bad conversion. Re-check Step 4: labels must be class 0 with")
    print("normalized xywh boxes.")

In [ ]:
from IPython.display import Image, display
display(Image(f"{R}/results.png"))

## Step 8 — Export the weights

Download `argus-plate-weights.zip` from the **Output** panel on the right, then
on the build machine:

```bash
cd ~/code/Argus
mkdir -p runs/detect/plate/weights
unzip ~/Downloads/argus-plate-weights.zip -d runs/detect/plate/weights
./.venv/bin/python ml/export_onnx.py --weights runs/detect/plate/weights/best.pt
```

`export_onnx.py` produces OpenVINO int8. Training was the only GPU step —
inference stays on the laptop CPU, which is why the export matters.

In [ ]:
!cd runs/detect/plate/weights && zip -q /kaggle/working/argus-plate-weights.zip best.pt last.pt
!ls -lh /kaggle/working/argus-plate-weights.zip

## If the session died mid-run

Kaggle kills long sessions. Re-run Steps 2, 3 and 5, then uncomment and run
this — it picks up from the last checkpoint rather than starting over.

In [ ]:
# !python ml/train_plate.py --epochs {EPOCHS} --device 0 --resume